# Stage-4 v2 K=4 — training (research record §44)

**Cell 4 runs a 3-step memory smoke test; cells 5–6 train Stage 4.** Pre-requisite: `stage4_v2_training_cache.ipynb` completed.
The one-time IDRiD test gate (cell 7) is disabled and must only be run once, explicitly, after training.

In [ ]:
# ==== [1] SETUP (Drive, repo, requirements incl. smp 0.5.0, TJDR/IDRiD paths) ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import stage4_v2_setup
SETUP_INFO = stage4_v2_setup.setup_stage4_v2(fetch_weights=True)

In [ ]:
# ==== [2] GPU + LOCAL COPY OF THE TRAINING CACHE ====
import shutil, torch
import stage4_v2_data as data
assert torch.cuda.is_available(), "GPU runtime required"
DRIVE_CACHE = data.cache_dir()
LOCAL_CACHE = "/content/cache_v2/Stage4Train/s4train-v1"
if not os.path.exists(os.path.join(LOCAL_CACHE, "manifest.json")):
    shutil.copytree(DRIVE_CACHE, LOCAL_CACHE, dirs_exist_ok=True)
tc = data.TrainingCache(LOCAL_CACHE, verify_files=True)
print({f"{d}/{r}": len(tc.names(r, d)) for d in ("IDRiD", "TJDR") for r in ("train", "val")}, tc.fingerprint)

In [ ]:
# ==== [3] CONFIGURATION (record §44; fixed before the first run) ====
import dataclasses, gc, json, posixpath
import colab_config
import stage4_v2_train as train
CFG = train.TrainConfig.from_defaults()
CFG.assert_fixed_design()
STAGE4_EXP = posixpath.join(colab_config.DRIVE.experiments_root, "Stage4V2")
os.makedirs(STAGE4_EXP, exist_ok=True)
print(dataclasses.asdict(CFG))

In [ ]:
# ==== [4] T4 MEMORY SMOKE TEST: batch 8 first, else batch 4 (balance kept: bs/2 IDRiD + bs/2 TJDR) ====
# 3 optimizer steps of the real model in a throw-away /content dir. Architecture and loss never change.
SMOKE_PATH = posixpath.join(STAGE4_EXP, "smoke_test.json")
if os.path.exists(SMOKE_PATH):
    SMOKE = json.load(open(SMOKE_PATH))
else:
    SMOKE = {"gpu": torch.cuda.get_device_name(0),
             "total_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "trials": {}}
    for bs in (8, 4):
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        smoke_dir = f"/content/stage4v2_smoke_bs{bs}"
        shutil.rmtree(smoke_dir, ignore_errors=True)
        try:
            train.run(smoke_dir, LOCAL_CACHE, max_steps=3,
                      cfg=dataclasses.replace(CFG, batch_size=bs, val_every=10**9, checkpoint_every=10**9))
            reserved = torch.cuda.max_memory_reserved() / 2**30
            trial = {"peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
                     "peak_reserved_gib": round(reserved, 2), "fits": reserved <= 0.85 * SMOKE["total_gib"]}
        except torch.cuda.OutOfMemoryError as exc:
            trial = {"oom": str(exc)[:300], "fits": False}
        SMOKE["trials"][str(bs)] = trial
        print("batch", bs, trial)
        if trial["fits"]:
            break
    fitting = [int(b) for b, t in SMOKE["trials"].items() if t["fits"]]
    assert fitting, "neither batch 8 nor batch 4 fits -- STOP (do not change architecture/loss)"
    SMOKE["selected_batch_size"] = max(fitting)
    json.dump(SMOKE, open(SMOKE_PATH, "w"), indent=1)
CFG = dataclasses.replace(CFG, batch_size=SMOKE["selected_batch_size"])
RUN_ID = f"stage4v2_k4_seed{CFG.seed}_bs{CFG.batch_size}"       # fixed: rerunning resumes the same run
RUN_DIR = posixpath.join(STAGE4_EXP, RUN_ID)
print(SMOKE); print("selected batch:", CFG.batch_size, "| run dir:", RUN_DIR)

In [ ]:
# ==== [5] TRAIN (resumable) -- THIS TRAINS ====
RESULT = train.run(RUN_DIR, LOCAL_CACHE, cfg=CFG)
print(RESULT)

In [ ]:
# ==== [6] EXPORT BEST EMA MODEL ====
MODEL_SHA = train.export_best(RUN_DIR)
print("Stage-4 v2 model sha256:", MODEL_SHA)

In [ ]:
# ==== [7] ONE-TIME IDRiD TEST GATE -- DISABLED. Run once, explicitly, after training. ====
RUN_ONE_TIME_GATE = False
if RUN_ONE_TIME_GATE:
    import pipeline_v2_config as v2cfg
    import stage4_v2_gate as gate
    MODEL_PATH = "<exported model.pt path>"
    GATE = gate.run_gate(MODEL_PATH, expected_sha256=MODEL_SHA, confirm=v2cfg.STAGE4_GATE_CONFIRM_TOKEN)
    print(GATE["decision"])